In [1]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [2]:
# Initialization

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
openai = OpenAI()

gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"

gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
MODEL= "gemini-3.1-flash-lite"

In [3]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [4]:
ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}

def get_ticket_price(destination_city):
    print(f"Tool called for city {destination_city}")
    price = ticket_prices.get(destination_city.lower(), "Unknown ticket price")
    return f"The price of a ticket to {destination_city} is {price}"

get_ticket_price("London")

Tool called for city London


'The price of a ticket to London is $799'

#### When we call an LLM to use tools, we have to tell it what tools it can use. And the way we tell it is with JSON. And there's just a fixed format, a type of JSON that you have to use to provide it, to tell it that

In [5]:
# There's a particular dictionary structure that's required to describe our function:

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}
print(type(price_function))

<class 'dict'>


In [6]:
# And this is included in a list of tools:

tools = [{"type": "function", "function": price_function}]
print(type(tools))
tools

<class 'list'>


[{'type': 'function',
  'function': {'name': 'get_ticket_price',
   'description': 'Get the price of a return ticket to the destination city.',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': 'The city that the customer wants to travel to'}},
    'required': ['destination_city'],
    'additionalProperties': False}}}]

#### We're going to have to detect if it's looking to run a tool. And if so, we need to run the tool, get the answer, and then send it back as a 2nd  message with the whole conversation history as a second call to the LM.

In [7]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    # 1st Call the Gemini API with the messages and our new item tools
    response = gemini.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    ## not return response.choices[0].message.content instead 
    # Check if the model decided to use a tool
    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        # If the model decided to use a tool, we can handle that tool call with our own function
        response = handle_tool_call(message)
        messages.append(message)
        messages.append(response)
        for message in messages:
            print(message)
        # 2nd Call the Gemini API with the tool's response so the model can process it
        response = gemini.chat.completions.create(model=MODEL, messages=messages)


    return response.choices[0].message.content

In [8]:
# We have to write that function handle_tool_call which will take the tool call message and return a response message with the tool's output.
def handle_tool_call(message):
    tool_call = message.tool_calls[0]
    if tool_call.function.name == "get_ticket_price":
        arguments = json.loads(tool_call.function.arguments)
        city = arguments.get('destination_city')
        price_details = get_ticket_price(city)
        response = {
            "role": "tool",
            "content": price_details,
            "tool_call_id": tool_call.id
        }
    return response

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

Handling multiple tool calls in 1 response. Say we asked

what are the ticket prices to London and Paris?

In [ ]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = gemini.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = gemini.chat.completions.create(model=MODEL, messages=messages)
    
    return response.choices[0].message.content

Problem is that we don't support a case where an LM wants to run multiple tools. To fix this in the method we handle tool call we replace if with fow and now it doesn't just take the first tool, but rather it iterates for tool call in message tool calls.

So if the LM asks to run multiple tools, we'll simply iterate through each one.

In [ ]:
def handle_tool_calls(message):
    responses = []
    # instean of handling only the first tool call, we can handle all tool calls in the message
    # so if is replaced with for loop
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

But if we make a promt like this "Please check ticket price to London. And if it's under $1,000, then please check Paris." I will still fail. 

The reason is because if you look at the  code, it only support one sequential, uh, set of tool calls. 
1. Call LLM
2. Check for Toll call
3 Send the response of toll to LLM for answer in 2nd LLM call

But now if we replace the if with while we get

1. Call LLM
2. Check for Toll call
3.  Keep checking the responses and keep calling tool if needed
4. If tool not needed call LLM for final response

It  now support multiple tool call requests in that first response, but once it comes back, it then makes another call without the tools.

In [ ]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = gemini.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = gemini.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    return response.choices[0].message.content

In [2]:
import sqlite3

In [3]:
DB = "prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)')
    conn.commit()

In [4]:
def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"

In [5]:
get_ticket_price("London")

DATABASE TOOL CALLED: Getting price for London


'No price data available for this city'

In [6]:
def set_ticket_price(city, price):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', (city.lower(), price, price))
        conn.commit()

In [ ]:
ticket_prices = {"london":799, "paris": 899, "tokyo": 1420, "sydney": 2999}
for city, price in ticket_prices.items():
    set_ticket_price(city, price)

In [ ]:
get_ticket_price("Tokyo")

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()